# Module 2.3: Attention Mechanisms

We finally have all the ingredients: we've turned words into embeddings, and we've mathematically injected their position in the sentence. 

Now, we build the engine of the Transformer: **The Attention Mechanism**. This is where the model learns context by allowing every word to physically "look" at every other word.

## 1. The Core Idea: Learnable Q, K, V

### The Concept
In Module 1, we learned the math for $Attention(Q, K, V) = softmax(Q K^T / \sqrt{d_k})V$. 

But where do $Q$ (Query), $K$ (Key), and $V$ (Value) come from? We don't just use the raw embeddings. Instead, we use Neural Network **Linear Layers** (Weight Matrices) to mathematically *project* the embedding into three different versions of itself.

### Why do we need it? (Learnable Parameters)
If we just used the raw embeddings, the Attention formula would always output the exact same similarities (e.g. "Apple" would always attend to "Fruit"). By using `Linear` layers, we give the model "knobs" it can turn. During training, the gradients adjust these Linear layers so the model **learns** *what* it should be querying for depending on the task (e.g., in a translation task, it might learn to query for verbs).

## 2. Self-Attention (Single Head)

### The Concept
"Self-Attention" means the sequence is attending to *itself*. Every word in the sentence asks every other word: "Are you relevant to my meaning?" 

For example, in the sentence "The bank of the river", the word `bank` will output a Query. The word `river` will have a Key that perfectly matches that Query. They will have a high dot product, and `bank` will absorb the Value vector from `river`—thereby updating its own meaning from "finance" to "nature".

### Why do we need it? (Parallel Context)
This replaces the old Recurrent Neural Networks (RNNs) which had to read left-to-right, one word at a time. Self-Attention evaluates the entire sentence synchronously using a single huge Matrix Multiplication. This allows for infinite reference distance without "forgetting" early words, and it runs blazing fast on a GPU.

In [1]:
import torch
import torch.nn as nn
import sys, os
sys.path.append(os.path.abspath("../"))
from src.functional import scaled_dot_product_attention

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        # These are the Learnable Weight Matrices!
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # x shape: (Batch, Seq_Len, d_model)
        Q = self.W_q(x)  # "What am I looking for?"
        K = self.W_k(x)  # "What do I have?"
        V = self.W_v(x)  # "What information will I give you?"
        
        # Use our math function from Module 1!
        # Output shape is the exact same as input shape!
        output, weights = scaled_dot_product_attention(Q, K, V)
        return output

# 1 Batch, 5 Words, 128 Dimensional Embedding
dummy_sentence_embeddings = torch.randn(1, 5, 128)
single_head = SingleHeadAttention(d_model=128)

contextualized_output = single_head(dummy_sentence_embeddings)
print(f"Input Shape : {dummy_sentence_embeddings.shape}")
print(f"Output Shape: {contextualized_output.shape}")

Input Shape : torch.Size([1, 5, 128])
Output Shape: torch.Size([1, 5, 128])


## 3. Multi-Head Attention (The Committee of Experts)

### The Analogy
Imagine reviewing a legal contract. If you review it alone, you might focus on the grammar but miss financial loopholes. 
Now imagine evaluating the contract with a **Committee of 8 Experts**. Expert 1 only checks grammar. Expert 2 only checks finances. Expert 3 tracks pronouns. 

**Multi-Head Attention (MHA)** is exactly this. Instead of one giant Attention layer, we split the 128 dimensions into 8 smaller "Heads" (each processing 16 dimensions). They all look at the sentence in parallel, and then we glue their findings back together at the end.

### Why do we need it? (Specialization)
Language is incredibly nuanced. If we only had one Single Head, the gradients would force the model to compromise and try to find a "middle ground" of attention. MHA allows the neural network to look at a single word from 8 (or 16, or 32) entirely different, specialized conceptual angles at the exact same time without compromising.

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head (e.g., 128 / 8 = 16)
        
        # We do one giant Linear layer for efficiency, then split it mathematically
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False) # The "Glue" at the end

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()
        
        # 1. Linear Projections
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 2. Split into Heads! 
        # (Batch, Seq_Len, d_model) -> (Batch, Seq_Len, Num_Heads, Head_Dim) -> (Batch, Num_Heads, Seq_Len, Head_Dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Apply Attention (Parallel over all heads!)
        output, _ = scaled_dot_product_attention(Q, K, V)
        
        # 4. Glue the Heads back together
        # Transpose back: (Batch, Num_Heads, Seq_Len, Head_Dim) -> (Batch, Seq_Len, Num_Heads, Head_Dim)
        # Contiguous/View: smashes Num_Heads and Head_Dim back into d_model
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        
        # 5. Final Output Projection
        return self.W_o(output)

# Look at the Dimensionality Tracking!
mha = MultiHeadAttention(d_model=128, num_heads=8)
final_output = mha(dummy_sentence_embeddings)

print(f"Input Shape : {dummy_sentence_embeddings.shape}")
print(f"Output Shape: {final_output.shape}")

Input Shape : torch.Size([1, 5, 128])
Output Shape: torch.Size([1, 5, 128])
